# Analysis and Visualisation

This notebook produces all figures and tables for the thesis results section.
No model training occurs here — all results are loaded from CSV files
saved by the experiment notebooks.

Run order:
1. 02_standard_protocol.ipynb
2. 03_crossview_protocol.ipynb
3. 04_ablation_study.ipynb
4. This notebook (05_analysis.ipynb)

Narrative structure:
- Act 1: Standard protocol results — who wins under ideal conditions?
- Act 2: Cross-view protocol results — how robust is each model to viewpoint shift?
- Act 3: WGA — where do models fail and is failure uniform or concentrated?
- Act 4: Ablation study — what drives the observed differences?
- Act 5: Synthesis — which paradigm wins and why?

In [ ]:
import os
import sys
import importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

repo_path = '/content/drive/MyDrive/BachelorsThesis'
results_path = f'{repo_path}/results'
figures_path = f'{repo_path}/results/figures'

os.makedirs(figures_path, exist_ok=True)

if not os.path.exists(repo_path):
    !git clone https://github.com/PurpleMono/BachelorsThesis.git {repo_path}
else:
    !git -C {repo_path} pull

sys.path.insert(0, repo_path)

!pip install anomalib==2.3.3 ADEval einops timm kornia -q

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

metrics = load_module("metrics", f"{repo_path}/evaluation/metrics.py")
wga_module = load_module("wga", f"{repo_path}/evaluation/wga.py")
vis = load_module("visualisation", f"{repo_path}/evaluation/visualisation.py")

compute_i_auroc = metrics.compute_i_auroc
compute_s_auroc = metrics.compute_s_auroc
compute_all_metrics = metrics.compute_all_metrics
compute_degradation_ratio = metrics.compute_degradation_ratio

wga_by_category = wga_module.wga_by_category
wga_by_viewpoint = wga_module.wga_by_viewpoint
wga_by_defect_type = wga_module.wga_by_defect_type
print_wga_summary = wga_module.print_wga_summary
find_disagreement_groups = wga_module.find_disagreement_groups

print("All modules loaded")

In [ ]:
# Load standard protocol scores
results_din_std = pd.read_csv(f'{results_path}/dinomaly_standard_scores.csv')
results_dino_std = pd.read_csv(f'{results_path}/anomalydino_standard_scores.csv')
results_inp_std = pd.read_csv(f'{results_path}/inpformer_standard_scores.csv')

# Load cross-view protocol scores
results_din_cv = pd.read_csv(f'{results_path}/dinomaly_crossview_scores.csv')
results_dino_cv = pd.read_csv(f'{results_path}/anomalydino_crossview_scores.csv')
results_inp_cv = pd.read_csv(f'{results_path}/inpformer_crossview_scores.csv')

# Load ablation summaries
abl1_summary = pd.read_csv(f'{results_path}/ablation1_volume_summary.csv')
abl2_summary = pd.read_csv(f'{results_path}/ablation2_singleclass_summary.csv')
abl3_summary = pd.read_csv(f'{results_path}/ablation3_compute_summary.csv')

# Dictionaries for convenience
df_dict_std = {
    'AnomalyDINO': results_dino_std,
    'Dinomaly': results_din_std,
    'INP-Former': results_inp_std,
}
df_dict_cv = {
    'AnomalyDINO': results_dino_cv,
    'Dinomaly': results_din_cv,
    'INP-Former': results_inp_cv,
}

print("All results loaded successfully")
print(f"Standard protocol — total test images per model: "
      f"{len(results_din_std)}")
print(f"Cross-view protocol — total test images per model: "
      f"{len(results_din_cv)}")

## Act 1: Standard Protocol

Research Question 1: Which DINOv2-based detection paradigm achieves the
highest anomaly detection performance under standard multi-view evaluation
conditions on Real-IAD?

We first establish the baseline performance of all three models under the
standard protocol — training on all five viewpoints and evaluating on the
full test set. This answers the fundamental question of which paradigm
performs best when deployment conditions match training conditions.

In [ ]:
print("Computing standard protocol metrics...")

std_metrics = {}
for model_name, df in df_dict_std.items():
    m = compute_all_metrics(df)
    std_metrics[model_name] = m

std_summary = pd.DataFrame(std_metrics).T.reset_index()
std_summary.columns = ['Model'] + list(std_summary.columns[1:])

print("\n" + "="*65)
print("TABLE 1: Standard Protocol Results — Real-IAD (All 30 Categories)")
print("="*65)
print(std_summary.round(4).to_string(index=False))

std_summary.to_csv(f'{results_path}/table1_standard_summary.csv',
                   index=False)
print(f"\nSaved to results/table1_standard_summary.csv")

In [ ]:
vis.plot_score_distributions(
    df_dict_std,
    output_path=f'{figures_path}/fig1_score_distributions_standard.png'
)
print("Figure 1 saved: score distributions (standard protocol)")

In [ ]:
vis.plot_per_category_comparison(
    df_dict_std,
    metric_fn=compute_i_auroc,
    metric_name='I-AUROC',
    output_path=f'{figures_path}/fig2_per_category_standard.png'
)
print("Figure 2 saved: per-category I-AUROC comparison")

In [ ]:
vis.plot_wga_category_table(
    df_dict_std,
    output_path=f'{figures_path}/fig3_wga_category_table_standard.png'
)
print("Figure 3 saved: WGA category table (standard protocol)")

In [ ]:
vis.plot_wga_heatmap_viewpoint(
    df_dict_std,
    output_path=f'{figures_path}/fig4_wga_heatmap_viewpoint_standard.png'
)
print("Figure 4 saved: WGA category x viewpoint heatmap (standard protocol)")

In [ ]:
# Top 5 hardest defect types per model on standard protocol
print("="*60)
print("HARDEST DEFECT TYPES — Standard Protocol")
print("="*60)

for model_name, df in df_dict_std.items():
    wga_df = wga_by_defect_type({model_name: df})
    if wga_df.empty:
        continue
    worst = wga_df.nsmallest(5, 'auroc')[
        ['defect_type', 'n_anomalous', 'auroc']]
    print(f"\n{model_name} — 5 hardest defect types:")
    print(worst.round(4).to_string(index=False))

# Save as CSV for thesis table
all_defect_wga = []
for model_name, df in df_dict_std.items():
    wga_df = wga_by_defect_type({model_name: df})
    if not wga_df.empty:
        wga_df['model'] = model_name
        all_defect_wga.append(wga_df)

if all_defect_wga:
    defect_table = pd.concat(all_defect_wga)
    defect_table.to_csv(
        f'{results_path}/table_defect_type_wga_standard.csv',
        index=False)
    print("\nSaved to results/table_defect_type_wga_standard.csv")

### Standard Protocol Ablations

Investigation 2: AnomalyDINO single-class vs multi-class
Investigation 3: Compute equalisation between Dinomaly and INP-Former

In [ ]:
print("="*60)
print("INVESTIGATION 2: AnomalyDINO Single-Class vs Multi-Class")
print("="*60)
print(abl2_summary.round(4).to_string(index=False))

vis.plot_ablation_singleclass(
    abl2_summary,
    output_path=f'{figures_path}/fig5_ablation_singleclass.png'
)
print("\nFigure 5 saved: AnomalyDINO single vs multi-class ablation")

In [ ]:
print("="*60)
print("INVESTIGATION 3: Compute Equalisation")
print("="*60)
print(abl3_summary.round(4).to_string(index=False))

vis.plot_ablation_compute(
    abl3_summary,
    output_path=f'{figures_path}/fig6_ablation_compute.png'
)
print("\nFigure 6 saved: compute equalisation ablation")

## Act 2: Cross-View Protocol

Research Question 2: How robust are DINOv2-based anomaly detection models
to viewpoint shift when deployed on camera angles not seen during training?

Having established baseline performance, we now introduce the cross-view
protocol — training on C1 and C2 only and evaluating on C3, C4, and C5.
This simulates a realistic deployment scenario where an inspection system
must generalise to new camera angles after initial commissioning.

The performance degradation ratio quantifies the cost of viewpoint shift
for each model and reveals whether the detection paradigm affects
robustness to distributional shift.

In [ ]:
print("Computing cross-view protocol metrics...")

cv_metrics = {}
for model_name, df in df_dict_cv.items():
    m = compute_all_metrics(df)
    cv_metrics[model_name] = m

cv_summary = pd.DataFrame(cv_metrics).T.reset_index()
cv_summary.columns = ['Model'] + list(cv_summary.columns[1:])

print("\n" + "="*65)
print("TABLE 2: Cross-View Protocol Results — Real-IAD")
print("="*65)
print(cv_summary.round(4).to_string(index=False))

cv_summary.to_csv(f'{results_path}/table2_crossview_summary.csv',
                  index=False)
print(f"\nSaved to results/table2_crossview_summary.csv")

In [ ]:
std_auroc = {m: compute_i_auroc(df) for m, df in df_dict_std.items()}
cv_auroc = {m: compute_i_auroc(df) for m, df in df_dict_cv.items()}

vis.plot_performance_comparison(
    results_std=std_auroc,
    results_cv=cv_auroc,
    metric='I-AUROC',
    output_path=f'{figures_path}/fig7_performance_comparison.png'
)
print("Figure 7 saved: standard vs cross-view performance comparison")

In [ ]:
degradation = {
    m: compute_degradation_ratio(std_auroc[m], cv_auroc[m])
    for m in std_auroc
}

print("\nPerformance Degradation Ratios:")
for model, deg in degradation.items():
    print(f"  {model}: {deg:.2f}%")

vis.plot_degradation_ratios(
    degradation,
    output_path=f'{figures_path}/fig8_degradation_ratios.png'
)
print("\nFigure 8 saved: degradation ratios")

In [ ]:
vis.plot_wga_heatmap_viewpoint(
    df_dict_cv,
    output_path=f'{figures_path}/fig9_wga_heatmap_viewpoint_crossview.png'
)
print("Figure 9 saved: WGA category x viewpoint (cross-view protocol)")

In [ ]:
# Compute per-category AUROC for both protocols
from evaluation.wga import compute_wga

delta_data = {}
for model_name in ['AnomalyDINO', 'Dinomaly', 'INP-Former']:
    std_wga = compute_wga(df_dict_std[model_name], ['category'])
    cv_wga = compute_wga(df_dict_cv[model_name], ['category'])

    std_cat = std_wga.set_index('category')['auroc']
    cv_cat = cv_wga.set_index('category')['auroc']
    delta = (cv_cat - std_cat).dropna()
    delta_data[model_name] = delta

delta_df = pd.DataFrame(delta_data)
delta_df = delta_df.sort_values(
    delta_df.columns[0], ascending=True)

print("\nPer-category AUROC delta (cross-view minus standard):")
print("Negative = performance drops under viewpoint shift")
print(delta_df.round(4).to_string())

delta_df.to_csv(
    f'{results_path}/table_delta_wga_crossview_vs_standard.csv')
print("\nSaved to results/table_delta_wga_crossview_vs_standard.csv")

In [ ]:
print("="*60)
print("INVESTIGATION 1: Training Data Volume Compensation")
print("="*60)
print(abl1_summary.round(4).to_string(index=False))

vis.plot_ablation_volume(
    abl1_summary,
    output_path=f'{figures_path}/fig10_ablation_volume.png'
)
print("\nFigure 10 saved: training volume compensation ablation")

## Act 3: Efficiency Analysis

Research Question 3: What are the computational trade-offs between the
three detection paradigms in terms of inference latency and memory footprint?

Industrial deployment imposes strict constraints on inference latency.
This section analyses the accuracy-efficiency trade-off for each model,
providing practical guidance for deployment decisions.

In [ ]:
# Load timing results if available, otherwise use placeholder
timing_file = f'{results_path}/inference_timing.csv'
if Path(timing_file).exists():
    timing_df = pd.read_csv(timing_file)
    timing_dict = dict(zip(timing_df['model'], timing_df['mean_ms']))
    memory_dict = dict(zip(timing_df['model'], timing_df['peak_mb']))
else:
    print("Timing results not yet available.")
    print("Run measure_inference_time() for each model in the")
    print("standard protocol notebook and save to results/inference_timing.csv")
    timing_dict = None

if timing_dict:
    auroc_dict = {m: compute_i_auroc(df)
                  for m, df in df_dict_std.items()}

    vis.plot_efficiency_tradeoff(
        timing_dict=timing_dict,
        auroc_dict=auroc_dict,
        output_path=f'{figures_path}/fig11_efficiency_tradeoff.png'
    )
    print("Figure 11 saved: accuracy vs inference time trade-off")

    # Accuracy to latency ratio
    print("\nAccuracy-to-Latency Ratio (I-AUROC / ms):")
    for model in ['AnomalyDINO', 'Dinomaly', 'INP-Former']:
        if model in timing_dict and model in auroc_dict:
            ratio = auroc_dict[model] / timing_dict[model]
            print(f"  {model}: {ratio:.4f}")

## Act 4: Synthesis

Bringing together all findings to answer the three research questions
and make a conclusive statement about which detection paradigm is most
suitable for multi-view industrial anomaly detection on Real-IAD.

In [ ]:
print("="*70)
print("COMBINED RESULTS SUMMARY")
print("="*70)

combined_rows = []
for model_name in ['AnomalyDINO', 'Dinomaly', 'INP-Former']:
    std_auroc_val = compute_i_auroc(df_dict_std[model_name])
    cv_auroc_val = compute_i_auroc(df_dict_cv[model_name])
    deg = compute_degradation_ratio(std_auroc_val, cv_auroc_val)
    s_auroc_val = compute_s_auroc(df_dict_std[model_name])

    combined_rows.append({
        'Model': model_name,
        'Paradigm': {
            'AnomalyDINO': 'Memory-Based',
            'Dinomaly': 'Reconstruction-Based',
            'INP-Former': 'Prototype-Based'
        }[model_name],
        'I-AUROC (Std)': round(std_auroc_val, 4),
        'S-AUROC (Std)': round(s_auroc_val, 4),
        'I-AUROC (CV)': round(cv_auroc_val, 4),
        'Degradation (%)': round(deg, 2),
    })

combined_df = pd.DataFrame(combined_rows)
print(combined_df.to_string(index=False))

combined_df.to_csv(
    f'{results_path}/table_combined_summary.csv', index=False)
print(f"\nSaved to results/table_combined_summary.csv")

In [ ]:
print("\n" + "="*60)
print("FIGURE INDEX — All thesis figures")
print("="*60)

figures = [
    ("fig1_score_distributions_standard.png",
     "Score distributions: normal vs anomalous (standard protocol)"),
    ("fig2_per_category_standard.png",
     "Per-category I-AUROC comparison (standard protocol)"),
    ("fig3_wga_category_table_standard.png",
     "WGA category table with colour scale (standard protocol)"),
    ("fig4_wga_heatmap_viewpoint_standard.png",
     "WGA heatmap: category x viewpoint (standard protocol)"),
    ("fig5_ablation_singleclass.png",
     "Ablation 2: AnomalyDINO single-class vs multi-class"),
    ("fig6_ablation_compute.png",
     "Ablation 3: Dinomaly vs INP-Former compute equalisation"),
    ("fig7_performance_comparison.png",
     "I-AUROC: standard vs cross-view protocol"),
    ("fig8_degradation_ratios.png",
     "Performance degradation ratios under viewpoint shift"),
    ("fig9_wga_heatmap_viewpoint_crossview.png",
     "WGA heatmap: category x viewpoint (cross-view protocol)"),
    ("fig10_ablation_volume.png",
     "Ablation 1: training volume compensation"),
    ("fig11_efficiency_tradeoff.png",
     "Accuracy vs inference time trade-off"),
]

for fname, description in figures:
    full_path = f'{figures_path}/{fname}'
    exists = Path(full_path).exists()
    status = "READY" if exists else "PENDING"
    print(f"  [{status}] {fname}")
    print(f"         {description}")

print(f"\nAll figures saved to: {figures_path}")
print("Insert figures into Word thesis using Insert > Picture")